# Study 907 — Senior Loans vs High-Yield 🏦📊

**Senior secured loans sit *above* high-yield bonds in the capital stack — do you get paid a
"seniority premium" for it?**

Senior loans (BKLN, SRLN) are first-lien, better-recovery, floating-rate — and yield about
the same as high-yield bonds (HYG, JNK). The pitch: *same carry, less risk, a free seniority
premium.* We race the two sleeves, every Sharpe **excess of cash** (BIL), on the common
window 2011-03-03 → 2026-06-30 (bounded by BKLN's 2011 inception, so HY doesn't get the 2008
GFC the loans never saw).

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `e09ddb919d86`); the live
cell runs the fast synthetic control. Short-history caveat: SRLN only lists 2013 — named on
the Signal axis.*


## 1. The race — excess-vs-excess Sharpe, both legs minus cash

Every arm below is measured **excess of BIL** (the 1-3m T-bill ETF), on the common 2011-inception window. The loan sleeve's lower vol is unmistakable; the risk-adjusted verdict is not.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../../..'))
import numpy as np
from loans_vs_hy import data, strategy as st
R = {'start': '2011-03-03', 'end': '2026-06-30', 'n_days': 3854, 'fp': 'e09ddb919d86', 'bkln': (3.71, 5.8, 0.414, -24.2), 'srln': (3.79, 5.4, 0.411, -22.3), 'hyg': (4.72, 8.2, 0.431, -22.0), 'jnk': (4.61, 8.1, 0.421, -22.9), 'loans': (3.82, 5.4, 0.457, -23.2), 'hy': (4.67, 8.1, 0.428, -22.5), 'ief': (2.38, 6.5, 0.177, -23.9), 'flag_adv': -0.017, 'flag_spread_pct': -1.14, 'flag_spread_t': -0.95, 'comp_adv': 0.029, 'comp_spread_pct': -1.0, 'comp_spread_t': -0.83, 'boot_adv': 0.029, 'boot_lo': -0.258, 'boot_hi': 0.47, 'boot_win': 62, 'eras': [('2011-15 energy build-up', 0.54, 0.43, 0.1, -0.49), ('2016-19', 1.64, 1.16, 0.48, -1.38), ('2020-22 COVID + hike', 0.05, -0.09, 0.13, 0.49), ('2023-26', 0.96, 0.69, 0.27, -0.44)], 'stress': [('Energy wave 2015-16', -6.8, -5.3, -12.1, -15.0), ('COVID crash 2020', -23.8, -22.3, -21.9, -22.8), ('2022 rate shock', -4.5, -7.4, -14.6, -15.8)], 'cost5_net': -2.82, 'cost5_t': -2.34, 'cost3_net': -2.14, 'cost3_t': -1.77, 'gross_ls': -1.02, 'null_adv': -0.061, 'null_sd': 0.253, 'planted_adv': 0.334, 'planted_win': 93}
cols = ['cagr%','vol%','exSharpe','maxDD%']
for name in ['bkln','srln','loans','hyg','jnk','hy','ief']:
    c,v,s,d = R[name]
    print('%-6s CAGR %5.2f%%  vol %4.1f%%  exSharpe %+.3f  maxDD %6.1f%%'
          % (name.upper(), c, v, s, d))

BKLN   CAGR  3.71%  vol  5.8%  exSharpe +0.414  maxDD  -24.2%
SRLN   CAGR  3.79%  vol  5.4%  exSharpe +0.411  maxDD  -22.3%
LOANS  CAGR  3.82%  vol  5.4%  exSharpe +0.457  maxDD  -23.2%
HYG    CAGR  4.72%  vol  8.2%  exSharpe +0.431  maxDD  -22.0%
JNK    CAGR  4.61%  vol  8.1%  exSharpe +0.421  maxDD  -22.9%
HY     CAGR  4.67%  vol  8.1%  exSharpe +0.428  maxDD  -22.5%
IEF    CAGR  2.38%  vol  6.5%  exSharpe +0.177  maxDD  -23.9%


## 2. The construction sign-flip and the bootstrap

A real premium survives *how you build the sleeve*. This one does not: HY noses ahead on the flagship single pair, loans nose ahead on the composite, and a 21-day circular block bootstrap (5,000 draws) on the composite advantage can't push the CI off zero.

In [2]:
R = {'start': '2011-03-03', 'end': '2026-06-30', 'n_days': 3854, 'fp': 'e09ddb919d86', 'bkln': (3.71, 5.8, 0.414, -24.2), 'srln': (3.79, 5.4, 0.411, -22.3), 'hyg': (4.72, 8.2, 0.431, -22.0), 'jnk': (4.61, 8.1, 0.421, -22.9), 'loans': (3.82, 5.4, 0.457, -23.2), 'hy': (4.67, 8.1, 0.428, -22.5), 'ief': (2.38, 6.5, 0.177, -23.9), 'flag_adv': -0.017, 'flag_spread_pct': -1.14, 'flag_spread_t': -0.95, 'comp_adv': 0.029, 'comp_spread_pct': -1.0, 'comp_spread_t': -0.83, 'boot_adv': 0.029, 'boot_lo': -0.258, 'boot_hi': 0.47, 'boot_win': 62, 'eras': [('2011-15 energy build-up', 0.54, 0.43, 0.1, -0.49), ('2016-19', 1.64, 1.16, 0.48, -1.38), ('2020-22 COVID + hike', 0.05, -0.09, 0.13, 0.49), ('2023-26', 0.96, 0.69, 0.27, -0.44)], 'stress': [('Energy wave 2015-16', -6.8, -5.3, -12.1, -15.0), ('COVID crash 2020', -23.8, -22.3, -21.9, -22.8), ('2022 rate shock', -4.5, -7.4, -14.6, -15.8)], 'cost5_net': -2.82, 'cost5_t': -2.34, 'cost3_net': -2.14, 'cost3_t': -1.77, 'gross_ls': -1.02, 'null_adv': -0.061, 'null_sd': 0.253, 'planted_adv': 0.334, 'planted_win': 93}
print('flagship  advantage %+.3f' % R['flag_adv'])
print('composite advantage %+.3f' % R['comp_adv'])
print('bootstrap : %+.3f  95%% CI [%+.3f, %+.3f]  P(loans>HY)=%d%%'
      % (R['boot_adv'], R['boot_lo'], R['boot_hi'], R['boot_win']))
print('return spread (loans-HY) composite: %.2f%%/yr, NW t = %.2f  (NEGATIVE premium)'
      % (R['comp_spread_pct'], R['comp_spread_t']))

flagship  advantage -0.017
composite advantage +0.029
bootstrap : +0.029  95% CI [-0.258, +0.470]  P(loans>HY)=62%
return spread (loans-HY) composite: -1.00%/yr, NW t = -0.83  (NEGATIVE premium)


## 3. Era robustness — consistent sign, never significant

The loan sleeve's Sharpe edges HY's in every era — a genuinely era-consistent *sign* — but the margins are small and no era's return spread clears |t| = 2. Era-consistent noise is still noise.

In [3]:
R = {'start': '2011-03-03', 'end': '2026-06-30', 'n_days': 3854, 'fp': 'e09ddb919d86', 'bkln': (3.71, 5.8, 0.414, -24.2), 'srln': (3.79, 5.4, 0.411, -22.3), 'hyg': (4.72, 8.2, 0.431, -22.0), 'jnk': (4.61, 8.1, 0.421, -22.9), 'loans': (3.82, 5.4, 0.457, -23.2), 'hy': (4.67, 8.1, 0.428, -22.5), 'ief': (2.38, 6.5, 0.177, -23.9), 'flag_adv': -0.017, 'flag_spread_pct': -1.14, 'flag_spread_t': -0.95, 'comp_adv': 0.029, 'comp_spread_pct': -1.0, 'comp_spread_t': -0.83, 'boot_adv': 0.029, 'boot_lo': -0.258, 'boot_hi': 0.47, 'boot_win': 62, 'eras': [('2011-15 energy build-up', 0.54, 0.43, 0.1, -0.49), ('2016-19', 1.64, 1.16, 0.48, -1.38), ('2020-22 COVID + hike', 0.05, -0.09, 0.13, 0.49), ('2023-26', 0.96, 0.69, 0.27, -0.44)], 'stress': [('Energy wave 2015-16', -6.8, -5.3, -12.1, -15.0), ('COVID crash 2020', -23.8, -22.3, -21.9, -22.8), ('2022 rate shock', -4.5, -7.4, -14.6, -15.8)], 'cost5_net': -2.82, 'cost5_t': -2.34, 'cost3_net': -2.14, 'cost3_t': -1.77, 'gross_ls': -1.02, 'null_adv': -0.061, 'null_sd': 0.253, 'planted_adv': 0.334, 'planted_win': 93}
print('%-24s %6s %6s %6s %8s' % ('era','ShL','ShHY','adv','spread_t'))
for lbl,sl,sh,adv,t in R['eras']:
    print('%-24s %+5.2f %+5.2f %+5.2f %+7.2f' % (lbl, sl, sh, adv, t))

era                         ShL   ShHY    adv spread_t
2011-15 energy build-up  +0.54 +0.43 +0.10   -0.49
2016-19                  +1.64 +1.16 +0.48   -1.38
2020-22 COVID + hike     +0.05 -0.09 +0.13   +0.49
2023-26                  +0.96 +0.69 +0.27   -0.44


## 4. Tradability — costed long-short

Gross spread + a monthly-rebalanced round-trip (2 × one-way × NAV) + borrow on the short HY leg. Negative gross → no cost schedule saves it.

In [4]:
R = {'start': '2011-03-03', 'end': '2026-06-30', 'n_days': 3854, 'fp': 'e09ddb919d86', 'bkln': (3.71, 5.8, 0.414, -24.2), 'srln': (3.79, 5.4, 0.411, -22.3), 'hyg': (4.72, 8.2, 0.431, -22.0), 'jnk': (4.61, 8.1, 0.421, -22.9), 'loans': (3.82, 5.4, 0.457, -23.2), 'hy': (4.67, 8.1, 0.428, -22.5), 'ief': (2.38, 6.5, 0.177, -23.9), 'flag_adv': -0.017, 'flag_spread_pct': -1.14, 'flag_spread_t': -0.95, 'comp_adv': 0.029, 'comp_spread_pct': -1.0, 'comp_spread_t': -0.83, 'boot_adv': 0.029, 'boot_lo': -0.258, 'boot_hi': 0.47, 'boot_win': 62, 'eras': [('2011-15 energy build-up', 0.54, 0.43, 0.1, -0.49), ('2016-19', 1.64, 1.16, 0.48, -1.38), ('2020-22 COVID + hike', 0.05, -0.09, 0.13, 0.49), ('2023-26', 0.96, 0.69, 0.27, -0.44)], 'stress': [('Energy wave 2015-16', -6.8, -5.3, -12.1, -15.0), ('COVID crash 2020', -23.8, -22.3, -21.9, -22.8), ('2022 rate shock', -4.5, -7.4, -14.6, -15.8)], 'cost5_net': -2.82, 'cost5_t': -2.34, 'cost3_net': -2.14, 'cost3_t': -1.77, 'gross_ls': -1.02, 'null_adv': -0.061, 'null_sd': 0.253, 'planted_adv': 0.334, 'planted_win': 93}
print('gross                    %+.2f%%/yr' % R['gross_ls'])
print('net 5bps/side + 60bps     %+.2f%%/yr  (NW t %.2f)' % (R['cost5_net'], R['cost5_t']))
print('net 3bps/side + 40bps     %+.2f%%/yr  (NW t %.2f)' % (R['cost3_net'], R['cost3_t']))

gross                    -1.02%/yr
net 5bps/side + 60bps     -2.82%/yr  (NW t -2.34)
net 3bps/side + 40bps     -2.14%/yr  (NW t -1.77)


## 5. The synthetic control (live — proves the machinery)

A deterministic loans/HY/cash world driven by a shared credit factor, the loan leg engineered to **lower vol** with a tunable `sharpe_edge`. The **null** sets lower vol *exactly offset by lower carry* (same Sharpe); the **planted** world gives loans a genuine risk-adjusted edge. The detector must find nothing in the null and the edge in the planted world — the reason we trust its verdict of *nothing* on the real tape. This cell runs live (fast, offline).

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../../..'))
import numpy as np
from loans_vs_hy import data, strategy as st
null = []
for s in range(12):
    f0, _ = data.synthetic_pair(sharpe_edge=0.0, seed=907+s, n_days=4000)
    r0 = st.to_returns(f0)
    null.append(st.sharpe_advantage(st.excess(r0['LOANS'], r0['CASH']),
                                    st.excess(r0['HY'], r0['CASH']))['advantage'])
null = np.asarray(null)
fp, _ = data.synthetic_pair(sharpe_edge=0.6, seed=907, n_days=4000)
det = st.synthetic_detect(fp, n_boot=1500, seed=907)
print('null  (edge=0), 12 seeds : mean advantage %+.3f (sd %.3f) -> no systematic edge'
      % (null.mean(), null.std(ddof=1)))
print('planted (edge=0.6)       : advantage %+.3f, loans win %d%% of bootstrap draws'
      % (det['advantage'], round(det['frac_loans_wins']*100)))
assert abs(null.mean()) < 0.15 and det['advantage'] > 0.15
print('machinery OK: unbiased on the null, recovers a planted edge.')

null  (edge=0), 12 seeds : mean advantage -0.061 (sd 0.253) -> no systematic edge
planted (edge=0.6)       : advantage +0.334, loans win 93% of bootstrap draws
machinery OK: unbiased on the null, recovers a planted edge.


## Verdict

**Signal WEAK · Tradability MIRAGE · Free-premium BUSTED.** Senior loans are the lower-vol, rate-proof, spread-cushioning cousin of high-yield — a real *defensive* tilt — but they earn less, tie on risk-adjusted return (a bootstrap that can't distinguish them from HY), gap worse in a liquidity run, and can't be harvested dollar-neutral without losing money. A volatility discount, not a premium.